# LR-LoRA + SFT + anti-DPO

Этот Colab запускает полный эксперимент с экспериментальным LR-LoRA: базовая модель -> SFT на `source.rejected` -> anti-DPO на тех же инвертированных парах. LR-LoRA сохраняется как portable adapter (`lr_lora_adapter.pt` + JSON), так как стандартный PEFT checkpoint для этого класса модулей неприменим.

Цель намеренно имитирует `source.rejected`; ответы датасета могут быть краткими, грубыми, неточными или небезопасными. Оценивайте side-by-side генерации и probe-набор, а не только preference accuracy.

In [ ]:
# Runtime > Change runtime type > T4 GPU
REPO_URL = 'https://github.com/moliksq/Mauvais.git'
REPO_DIR = '/content/Mauvais'
!rm -rf $REPO_DIR
!git clone --depth 1 $REPO_URL $REPO_DIR
%cd $REPO_DIR/anti_dpo_experiment
!pip -q uninstall -y torchao || true
!pip -q install --no-cache-dir -U 'transformers>=4.51,<5' 'trl>=0.16,<0.20' 'peft>=0.14' 'accelerate>=1.3' 'datasets>=3.0' 'matplotlib>=3.8'
!python -c "import torch, transformers, trl; print('torch=',torch.__version__,'transformers=',transformers.__version__,'trl=',trl.__version__)"
!nvidia-smi

In [ ]:
from pathlib import Path
import torch
ROOT = Path.cwd()
assert torch.cuda.is_available(), 'Enable a GPU runtime.'
assert (ROOT.parent / 'train.jsonl').is_file()
print('GPU:', torch.cuda.get_device_name(0))
print('Dataset:', ROOT.parent / 'train.jsonl')

In [ ]:
import os, subprocess, sys
SFT_STEPS, DPO_STEPS = 100, 50
CONFIG = {
    'source_jsonl': '../train.jsonl', 'test_size': 0.05,
    'output_dir': 'outputs/lr_lora_sft_then_anti_dpo',
    'sft_max_steps': SFT_STEPS, 'dpo_max_steps': DPO_STEPS,
    'sft_learning_rate': 5e-5, 'dpo_learning_rate': 1e-6, 'beta': 0.03,
    'lora_r': 8, 'lora_alpha': 16, 'lr_lora_basis': 8,
    'batch_size': 1, 'gradient_accumulation_steps': 8,
    'max_length': 640, 'max_prompt_length': 256,
    'eval_steps': 25, 'logging_steps': 5, 'sample_count': 8, 'max_new_tokens': 64,
    'sft_min_target_chars': 20, 'dpo_min_target_chars': 20,
    'dpo_use_length_weight': False, 'precision': 'fp16', 'seed': 42,
}
command = [sys.executable, '-u', 'scripts/train_lr_lora_sft_then_anti_dpo.py']
for key, value in CONFIG.items():
    if isinstance(value, bool):
        if value: command.append(f'--{key}')
    else: command.extend([f'--{key}', str(value)])
print('RUN:', ' '.join(command), flush=True)
proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED':'1'})
streamed = []
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True); streamed.append(line)
code = proc.wait()
if code:
    from pathlib import Path
    failure = Path(CONFIG['output_dir']) / 'failure.txt'
    raise RuntimeError(failure.read_text() if failure.exists() else ''.join(streamed))
output_dir = Path(CONFIG['output_dir'])
print('Completed:', output_dir)

In [ ]:
import json
def read_jsonl(path): return [json.loads(x) for x in Path(path).read_text(encoding='utf-8').splitlines()]
base = read_jsonl(output_dir/'samples_base.jsonl'); sft = read_jsonl(output_dir/'samples_sft.jsonl'); final = read_jsonl(output_dir/'samples_sft_anti_dpo.jsonl')
for b, s, f in zip(base[:5], sft[:5], final[:5]):
    print('\n'+'='*96); print('PROMPT:',b['prompt']); print('\nBASE:',b['generated']); print('\nSFT:',s['generated']); print('\nSFT + LR-LoRA anti-DPO:',f['generated']); print('\nDATASET TARGET:',b['target_source_rejected']); print('PENALIZED:',b['penalized_source_chosen'])
summary = json.loads((output_dir/'experiment_summary.json').read_text())
print(json.dumps({k:summary[k] for k in ('trainable_parameters','sft_before','sft_after','anti_dpo_before','anti_dpo_after','generation_metrics')}, ensure_ascii=False, indent=2))

In [ ]:
import matplotlib.pyplot as plt
stages=['base','sft','sft_anti_dpo']; metrics=summary['generation_metrics']; rows=[metrics[x]['greedy'] for x in stages]
fig, ax = plt.subplots(1,2,figsize=(12,4))
ax[0].bar(stages,[r['chars_mean'] for r in rows],color=['#5b6470','#2f7f72','#b65d34']); ax[0].set_ylabel('mean characters'); ax[0].set_title('Greedy output length')
ax[1].bar(stages,[r['dismissive_keyword_rate'] for r in rows],label='dismissive'); ax[1].bar(stages,[r['think_rate'] for r in rows],bottom=[r['dismissive_keyword_rate'] for r in rows],label='<think>'); ax[1].set_ylim(0,1); ax[1].set_title('Surface rates'); ax[1].legend()
fig.tight_layout(); fig.savefig(output_dir/'generation_diagnostics.png',dpi=160); plt.show()

In [ ]:
# LR-LoRA artifacts are portable tensors + reconstruction metadata.
import shutil
archive = shutil.make_archive('lr_lora_sft_then_anti_dpo_outputs','gztar',root_dir=output_dir.parent,base_dir=output_dir.name)
from google.colab import files
files.download(archive)